# Image PreProcess Check

In [ ]:
import rasterio

path = "/Volumes/Windows8_OS/Dataset/Dataset-OG/Images/Oil/00000.tif"

with rasterio.open(path) as src:
    print("Bands:", src.count)
    print("Dtype:", src.dtypes)
    print("CRS:", src.crs)
    print("Resolution:", src.res)


In [ ]:
import numpy as np
import rasterio

with rasterio.open(path) as src:
    vv = src.read(1)

print("Min:", np.nanmin(vv))
print("Max:", np.nanmax(vv))


In [ ]:
import matplotlib.pyplot as plt

plt.hist(vv.flatten(), bins=200)
plt.title("VV Histogram")
plt.show()


In [ ]:
plt.imshow(vv, cmap="gray")
plt.colorbar()
plt.title("Raw VV Display")
plt.show()


In [ ]:
with rasterio.open(path) as src:
    print(src.descriptions)


In [ ]:
def is_preprocessed_sar(img):
    if np.nanmin(img) < -5:
        return "ALREADY PREPROCESSED (dB)"
    if np.nanmax(img) > 100:
        return "RAW INTENSITY (NOT preprocessed)"
    return "UNKNOWN / PARTIAL"

print(is_preprocessed_sar(vv))


# Conversion from 2 bands to 1

In [ ]:
import rasterio
import numpy as np

with rasterio.open(path) as src:
    vv = src.read(1).astype(np.float32)
    vh = src.read(2).astype(np.float32)

In [ ]:
def normalize(img, pmin=2, pmax=98):
    lo, hi = np.percentile(img, (pmin, pmax))
    img = np.clip(img, lo, hi)
    return (img - lo) / (hi - lo + 1e-6)

vv_n = normalize(vv)
vh_n = normalize(vh)

# Model

In [ ]:
import os
import tensorflow as tf
import numpy as np
import rasterio
from tqdm import tqdm

In [ ]:
DATASET_ROOT = "/Volumes/Windows8_OS/Dataset/Dataset-OG"
IMAGES_ROOT = os.path.join(DATASET_ROOT, "Images")

OIL_DIR = os.path.join(IMAGES_ROOT, "Oil")
NO_OIL_DIR = os.path.join(IMAGES_ROOT, "No_Oil")

# Break

# Performance Optimizations Applied

1. **Pre-allocated arrays**: Faster than appending to lists
2. **Efficient normalization**: Handles NaN values properly
3. **TensorFlow dataset pipelines**: Parallel loading with AUTOTUNE
4. **Batch prefetching**: Overlaps data loading with training
5. **Early stopping**: Prevents overfitting and saves time
6. **Learning rate reduction**: Adapts to training progress

**Option 1**: Best for smaller datasets (loads all into memory)  
**Option 2**: Best for larger datasets (streams data on-demand)

In [ ]:
def normalize(img, pmin=2, pmax=98):
    """Fast normalization using percentile clipping."""
    # Handle NaN values
    valid_data = img[~np.isnan(img)]
    
    if len(valid_data) == 0:
        return img
    
    lo, hi = np.percentile(valid_data, (pmin, pmax))
    
    # Clip and normalize
    img_clipped = np.clip(img, lo, hi)
    
    # Avoid division by zero
    range_val = hi - lo
    if range_val < 1e-6:
        return np.zeros_like(img)
    
    return (img_clipped - lo) / range_val

In [ ]:
def load_sar_tiff(path):
    with rasterio.open(path) as src:
        vv = src.read(1).astype(np.float32)  # Band 1 = VV
        vh = src.read(2).astype(np.float32)  # Band 2 = VH

    vv = normalize(vv)
    vh = normalize(vh)

    x = np.stack([vv, vh], axis=-1)  # (H, W, 2)
    return x

In [ ]:
def load_detection_dataset(oil_dir, no_oil_dir, max_samples=None):
    """Load SAR images efficiently with optional sample limit."""
    oil_files = [os.path.join(oil_dir, f) for f in os.listdir(oil_dir) if f.endswith(".tif")]
    no_oil_files = [os.path.join(no_oil_dir, f) for f in os.listdir(no_oil_dir) if f.endswith(".tif")]
    
    # Optional: limit samples for faster testing
    if max_samples:
        oil_files = oil_files[:max_samples]
        no_oil_files = no_oil_files[:max_samples]

    print(f"Oil images: {len(oil_files)}")
    print(f"No-Oil images: {len(no_oil_files)}")
    
    total_samples = len(oil_files) + len(no_oil_files)
    
    # Pre-allocate arrays for better performance
    X = np.zeros((total_samples, 512, 512, 2), dtype=np.float32)
    y = np.zeros(total_samples, dtype=np.int32)
    
    idx = 0
    
    for path in tqdm(oil_files, desc="Loading OIL"):
        X[idx] = load_sar_tiff(path)
        y[idx] = 1
        idx += 1
    print("Oil Loaded")

    for path in tqdm(no_oil_files, desc="Loading NO-OIL"):
        X[idx] = load_sar_tiff(path)
        y[idx] = 0
        idx += 1
    print("No Oil Loaded")

    return X, y


In [ ]:
X, y = load_detection_dataset(OIL_DIR, NO_OIL_DIR)

print("Done")

In [ ]:
print("X shape:", X.shape)  # (N, 512, 512, 2)
print("y shape:", y.shape)  # (N,)

In [ ]:
from sklearn.model_selection import train_test_split

# Split data: 66% training, 34% validation
X_train, X_val, y_train, y_val = train_test_split(
    X, y, 
    test_size=0.34, 
    random_state=42,
    stratify=y  # Ensure balanced split
)

print(f"Training samples: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Validation samples: {len(X_val)} ({len(X_val)/len(X)*100:.1f}%)")
print(f"Training - Oil: {np.sum(y_train)}, No-Oil: {len(y_train) - np.sum(y_train)}")
print(f"Validation - Oil: {np.sum(y_val)}, No-Oil: {len(y_val) - np.sum(y_val)}")

# 2nd Option

# Train/Validation Split Configuration

**Split Ratio:**
- Training: 66%
- Validation: 34%

This split ensures:
- Sufficient training data for model learning
- Adequate validation data for monitoring generalization
- Stratified split maintains class balance in both sets

In [ ]:
def load_sar_tiff_tf(path):
    def _read(path_str):
        with rasterio.open(path_str.decode()) as src:
            vv = src.read(1).astype(np.float32)
            vh = src.read(2).astype(np.float32)

        vv = np.clip(vv, -35.0, 5.0)
        vh = np.clip(vh, -40.0, 0.0)

        vv = (vv + 35.0) / 40.0
        vh = (vh + 40.0) / 40.0

        # 🔑 FORCE float32
        out = np.stack([vv, vh], axis=-1).astype(np.float32)
        return out

    img = tf.numpy_function(_read, [path], tf.float32)
    img.set_shape([512, 512, 2])
    return img

In [ ]:
oil_files = [os.path.join(OIL_DIR, f) for f in os.listdir(OIL_DIR) if f.endswith(".tif")]
no_oil_files = [os.path.join(NO_OIL_DIR, f) for f in os.listdir(NO_OIL_DIR) if f.endswith(".tif")]

In [ ]:
oil_paths = tf.data.Dataset.from_tensor_slices(oil_files)
no_oil_paths = tf.data.Dataset.from_tensor_slices(no_oil_files)

In [ ]:
oil_ds = oil_paths.map(
    lambda x: (load_sar_tiff_tf(x), 1),
    num_parallel_calls=tf.data.AUTOTUNE
)

In [ ]:
no_oil_ds = no_oil_paths.map(
    lambda x: (load_sar_tiff_tf(x), 0),
    num_parallel_calls=tf.data.AUTOTUNE
)

In [ ]:
BATCH_SIZE = 8

# Combine datasets with labels
dataset = oil_ds.concatenate(no_oil_ds)

# Get total size for splitting
total_size = len(oil_files) + len(no_oil_files)
train_size = int(0.66 * total_size)

# Shuffle the entire dataset
dataset = dataset.shuffle(buffer_size=total_size, seed=42, reshuffle_each_iteration=False)

# Split into train and validation
train_ds = dataset.take(train_size)
val_ds = dataset.skip(train_size)

# Batch and prefetch for performance
train_ds = train_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print(f"Training batches: ~{train_size // BATCH_SIZE}")
print(f"Validation batches: ~{(total_size - train_size) // BATCH_SIZE}")

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

def build_detection_cnn():
    model = Sequential()

    # 6 Convolutional layers (32 filters each)
    for i in range(6):
        if i == 0:
            model.add(Conv2D(
                filters=32,
                kernel_size=(3, 3),
                activation='relu',
                padding='same',
                input_shape=(512, 512, 2)
            ))
        else:
            model.add(Conv2D(
                filters=32,
                kernel_size=(3, 3),
                activation='relu',
                padding='same'
            ))

        model.add(MaxPooling2D(pool_size=(2, 2)))

    model.add(Flatten())

    # Two hidden dense layers (20 neurons each)
    model.add(Dense(20, activation='relu'))
    model.add(Dense(20, activation='relu'))

    # Output layer
    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [ ]:
model = build_detection_cnn()

# Break

In [ ]:
# For Option-1: Using pre-loaded arrays with proper train/val split
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Callbacks for better training
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

history = model.fit(
    X_train, y_train,
    batch_size=8,
    epochs=50,
    validation_data=(X_val, y_val),
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

In [ ]:
# For Option-2: Using TensorFlow datasets with proper train/val split
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

# Last opt

In [19]:
import tensorflow as tf
import rasterio
import numpy as np

def load_sar_tiff_tf(path):
    def _read(p):
        with rasterio.open(p.decode()) as src:
            vv = src.read(1).astype(np.float32)
            vh = src.read(2).astype(np.float32)

        # Fixed-range normalization
        vv = np.clip(vv, -35.0, 5.0)
        vh = np.clip(vh, -40.0, 0.0)

        vv = (vv + 35.0) / 40.0
        vh = (vh + 40.0) / 40.0

        img = np.stack([vv, vh], axis=-1).astype(np.float32)

        # 🔑 RESIZE 2048 → 512
        img = tf.image.resize(img, (512, 512), method="bilinear").numpy()

        return img

    img = tf.numpy_function(_read, [path], tf.float32)
    img.set_shape([512, 512, 2])
    return img

In [20]:
import os

DATASET_ROOT = "/Volumes/Windows8_OS/Dataset/Dataset-OG"
IMAGES_ROOT = os.path.join(DATASET_ROOT, "Images")

OIL_DIR = os.path.join(IMAGES_ROOT, "Oil")
NO_OIL_DIR = os.path.join(IMAGES_ROOT, "No_Oil")

oil_files = [os.path.join(OIL_DIR, f) for f in os.listdir(OIL_DIR) if f.endswith(".tif")]
no_oil_files = [os.path.join(NO_OIL_DIR, f) for f in os.listdir(NO_OIL_DIR) if f.endswith(".tif")]

print("Oil images:", len(oil_files))
print("No-Oil images:", len(no_oil_files))

Oil images: 150
No-Oil images: 150


In [21]:
def make_dataset(paths, label, batch_size=8):
    ds = tf.data.Dataset.from_tensor_slices(paths)
    ds = ds.map(lambda x: (load_sar_tiff_tf(x), label),
                num_parallel_calls=tf.data.AUTOTUNE)
    return ds


In [22]:
oil_ds = make_dataset(oil_files, 1)
no_oil_ds = make_dataset(no_oil_files, 0)

dataset = oil_ds.concatenate(no_oil_ds)

total_samples = len(oil_files) + len(no_oil_files)

dataset = dataset.shuffle(
    buffer_size=total_samples,
    reshuffle_each_iteration=False
)

dataset = dataset.cache()
dataset = dataset.prefetch(tf.data.AUTOTUNE)

options = tf.data.Options()
options.experimental_deterministic = False
dataset = dataset.with_options(options)

# Split into 70-30

In [23]:
num_oil = len(oil_files)
num_no_oil = len(no_oil_files)

total_samples = num_oil + num_no_oil
print("Total samples:", total_samples)

Total samples: 300


In [24]:
dataset = dataset.shuffle(
    buffer_size=total_samples,
    reshuffle_each_iteration=False  # 🔑 important
)

In [25]:
train_size = int(0.8 * total_samples)
test_size  = total_samples - train_size

print("Train samples:", train_size)
print("Test samples:", test_size)

Train samples: 240
Test samples: 60


In [26]:
BATCH_SIZE = 8

train_ds = dataset.take(train_size)
test_ds  = dataset.skip(train_size)

train_ds = train_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds  = test_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [27]:
x, y = next(iter(train_ds))
print("Batch shape:", x.shape)

Batch shape: (8, 512, 512, 2)


# Model

In [28]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

def build_detection_cnn():
    model = Sequential()

    for i in range(6):
        if i == 0:
            model.add(Conv2D(
                32, (3, 3),
                activation='relu',
                padding='same',
                input_shape=(512, 512, 2)
            ))
        else:
            model.add(Conv2D(
                32, (3, 3),
                activation='relu',
                padding='same'
            ))

        model.add(MaxPooling2D(pool_size=(2, 2)))

    model.add(Flatten())
    model.add(Dense(20, activation='relu'))
    model.add(Dense(20, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [29]:
model = build_detection_cnn()
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 512, 512, 32)   │           608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 256, 256, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 256, 256, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 128, 128, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 128, 128, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 64, 64, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 32, 32, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 16, 16, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 8, 8, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 20)             │        40,980 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 20)             │           420 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            21 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 88,269 (344.80 KB)

 Trainable params: 88,269 (344.80 KB)

 Non-trainable params: 0 (0.00 B)

In [38]:
history = model.fit(train_ds, epochs=50)

Epoch 1/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 8s 256ms/step - accuracy: 0.4654 - loss: 180.3577
Epoch 2/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 8s 278ms/step - accuracy: 0.4208 - loss: 108.8450
Epoch 3/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 7s 245ms/step - accuracy: 0.6674 - loss: 591.9667
Epoch 4/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 9s 307ms/step - accuracy: 0.5607 - loss: 1631.9899
Epoch 5/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 13s 424ms/step - accuracy: 0.5218 - loss: 1417.0452
Epoch 6/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 8s 252ms/step - accuracy: 0.4458 - loss: 6634.7334
Epoch 7/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 8s 251ms/step - accuracy: 0.4136 - loss: 4227.0991
Epoch 8/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 10s 326ms/step - accuracy: 0.4025 - loss: 596.0852
Epoch 9/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 7s 238ms/step - accuracy: 0.4317 - loss: 515.6793
Epoch 10/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 7s 243ms/step - accuracy: 0.4011 - loss: 895.1832
Epoch 11/50
30/30 ━━━━━━━━━━━━━━━━━━━━ 8s 250ms/step - accuracy: 0.3920 - loss: 969.0708
Epoch 12/50
30/30 ━━━━━━

In [39]:
train_loss, train_acc = model.evaluate(train_ds, verbose=0)
print(f"Training Accuracy: {train_acc * 100:.2f}%")

Training Accuracy: 77.50%


In [40]:
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"Testing Accuracy: {test_acc * 100:.2f}%")

Testing Accuracy: 75.00%


In [41]:
print("===================================")
print(f"Train Accuracy: {train_acc * 100:.2f}%")
print(f"Test  Accuracy: {test_acc * 100:.2f}%")
print("===================================")

Train Accuracy: 77.50%
Test  Accuracy: 75.00%
